# CS4603 PA4 — Document Analyst

Development & testing notebook. Section headers match the tasks in `README.md`.
Fill in each cell, run everything top-to-bottom, and **keep all outputs visible** before submitting.
Record explanations and analysis answers in `STUDENT_ANALYSIS.md`.


## Part 0 — Setup & Corpus Ingestion
Env config + ingest `data/annual_report.pdf` into Databricks Vector Search (Task 0.3).


In [0]:
# TODO(0.1): load config / verify env vars
# from config import ...



In [0]:
display(dbutils.fs.ls("/Volumes/main/default/pa4/"))

In [0]:
spark.sql("SHOW FUNCTIONS").filter("function like '%ai_parse%'").show()

In [0]:
%pip install databricks-vectorsearch

In [0]:
dbutils.library.restartPython()

In [0]:
# TODO(0.3): ingest corpus -> Delta table -> Vector Search index; wait until READY
# from rag.ingest import ingest
# ingest(spark, volume_path='/Volumes/main/default/pa4/annual_report.pdf')

import os
from datetime import timedelta
from databricks.vector_search.client import VectorSearchClient

UC_CATALOG = "main"
UC_SCHEMA = "default"
EMBEDDINGS_ENDPOINT = "databricks-gte-large-en"
VECTOR_SEARCH_ENDPOINT = "mehmood-vs-endpoint" 
VECTOR_SEARCH_INDEX = "main.default.mehmood_analyst_index"

VOLUME_PATH = "/Volumes/main/default/pa4"
PDF_FILE = "annual_report.pdf"
FULL_PDF_PATH = f"{VOLUME_PATH}/{PDF_FILE}"

# Tables that will be created
PARSE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.pa4_parsed_documents"
CHUNK_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.pa4_chunks"

print("📋 Configuration")
print(f"   PDF         : {FULL_PDF_PATH}")
print(f"   Parse table : {PARSE_TABLE}")
print(f"   Chunk table : {CHUNK_TABLE}")
print(f"   VS endpoint : {VECTOR_SEARCH_ENDPOINT}")
print(f"   VS index    : {VECTOR_SEARCH_INDEX}\n")

try:
    files = dbutils.fs.ls(VOLUME_PATH)
    if not any(f.name == PDF_FILE for f in files):
        raise FileNotFoundError(f"{PDF_FILE} not found in {VOLUME_PATH}")
    print("✅ PDF found in volume.\n")
except Exception as e:
    print(f"❌ PDF not accessible: {e}")
    raise
print("1/4 Parsing PDF with ai_parse_document ...")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {PARSE_TABLE} (
        path STRING,
        parsed VARIANT
    ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"""
    INSERT OVERWRITE {PARSE_TABLE}
    SELECT
        path,
        ai_parse_document(content) AS parsed
    FROM read_files('{FULL_PDF_PATH}', format => 'binaryFile')
""")

parsed_count = spark.table(PARSE_TABLE).count()
print(f"   Parsed {parsed_count} document(s) into {PARSE_TABLE}.\n")

print("2/4 Chunking with ai_prep_search ...")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
        chunk_id STRING,
        chunk_to_retrieve STRING,
        chunk_to_embed STRING,
        source STRING,
        page STRING
    ) TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"""
    INSERT OVERWRITE {CHUNK_TABLE}
    SELECT
        chunk.value:chunk_id::STRING AS chunk_id,
        chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
        chunk.value:chunk_to_embed::STRING AS chunk_to_embed,
        prepped.path AS source,
        chunk.value:pages[0].page_id::STRING AS page
    FROM (
        SELECT path, ai_prep_search(parsed) AS result FROM {PARSE_TABLE}
    ) prepped,
        LATERAL variant_explode(prepped.result:document.contents) AS chunk
""")

chunk_count = spark.table(CHUNK_TABLE).count()
print(f"   Created {chunk_count} chunks in {CHUNK_TABLE}.\n")
print("3/4 Creating / verifying Vector Search resources ...")

client = VectorSearchClient()

existing_endpoints = [e["name"] for e in client.list_endpoints().get("endpoints", [])]
if VECTOR_SEARCH_ENDPOINT not in existing_endpoints:
    client.create_endpoint(name=VECTOR_SEARCH_ENDPOINT, endpoint_type="STANDARD")
    print(f"   Created endpoint '{VECTOR_SEARCH_ENDPOINT}'.")
else:
    print(f"   Endpoint '{VECTOR_SEARCH_ENDPOINT}' already exists.")

existing_indexes = [
    idx["name"] for idx in client.list_indexes(name=VECTOR_SEARCH_ENDPOINT).get("vector_indexes", [])
]
if VECTOR_SEARCH_INDEX not in existing_indexes:
    client.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_SEARCH_INDEX,
        source_table_name=CHUNK_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_to_retrieve",
        embedding_model_endpoint_name=EMBEDDINGS_ENDPOINT,
    )
    print(f"   Created index '{VECTOR_SEARCH_INDEX}'.")
else:
    print(f"   Index '{VECTOR_SEARCH_INDEX}' already exists.")

print("\n4/4 Waiting for index to become READY (this may take several minutes) ...")
index = client.get_index(endpoint_name=VECTOR_SEARCH_ENDPOINT, index_name=VECTOR_SEARCH_INDEX)
index.wait_until_ready(timeout=timedelta(seconds=1800))

test = index.similarity_search(
    query_text="What was the net revenue?",
    columns=["chunk_to_retrieve", "source"],
    num_results=1,
)
if test.get("result", {}).get("data_array"):
    print("✅ Index is READY and queryable.")
    sample_text = test["result"]["data_array"][0][0]
    print("   Sample chunk:", sample_text[:200], "...")
else:
    print("⚠️ Index READY but returned no results. Check your chunk table content.")

print("\n🎉 Ingestion complete. Your Vector Search index is ready for Part 1.")

In [0]:
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient()
index = client.get_index(
    endpoint_name="mehmood-vs-endpoint",
    index_name="main.default.mehmood_analyst_index"
)

# Try the similarity search directly
try:
    results = index.similarity_search(
        query_text="What was net revenue in 2023?",
        columns=["chunk_to_retrieve", "source"],
        num_results=2
    )
    data = results.get("result", {}).get("data_array", [])
    if data:
        print("✅ Index is working! Sample chunk:")
        print(data[0][0][:2000], "...")
    else:
        print("⚠️ No results returned.")
except Exception as e:
    print("❌ Error during search:", e)

## Part 1 — Build the Document Analyst graph
Nodes: planner (1.2), supervisor (1.3), RAG agent (1.4), MCP tools (1.5), synthesizer (1.6), full graph (1.7).


In [0]:
# TODO(1.7): build the compiled graph
# from agent.graph import build_graph
# graph = build_graph()



In [0]:
# TODO(1.7): visualize the compiled graph
# from IPython.display import Image
# Image(graph.get_graph().draw_mermaid_png())



### Test the graph


In [0]:
# Retrieval-only query
# graph.invoke({'messages':[{'role':'user','content':'What was the net income in 2023?'}]})



In [0]:
# Computation-only query
# graph.invoke({'messages':[{'role':'user','content':'What is 15% of 2.4 billion?'}]})



In [0]:
# Combined query — show the full step-by-step execution trace
# graph.invoke({'messages':[{'role':'user','content':'What was the revenue in 2023, and what would a 10% increase look like?'}]})



### Required — offline smoke test
Runs the graph with a mocked LLM (no Databricks calls). Same test Bonus A automates.


In [0]:
!python -m pytest tests/test_smoke.py -q


## Part 2 — Deployment
Package as an MLflow models-from-code model, register in Unity Catalog, create the serving endpoint (Tasks 2.1–2.4).
Reference: `databricks_deployment_v1/deployment.ipynb`.


In [0]:
# TODO(2.1): sanity-check the model definition imports cleanly
!python -c "import deployment.agent_model"



In [0]:
# TODO(2.2): log + register the model version in Unity Catalog



In [0]:
# TODO(2.3): create/update the serving endpoint; wait for READY; print the URL



### Test the deployed endpoint (Task 2.4)


In [0]:
# curl the endpoint and show the raw response



In [0]:
# Response shape depends on how you logged the model (see README Task 2.4 / GUIDE §7).
#
# Path A — raw LangGraph state (mlflow.langchain.log_model, v1 style):
# import requests
# url = f'{DATABRICKS_HOST}/serving-endpoints/<your-endpoint-name>/invocations'
# r = requests.post(url, headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}'},
#                   json={'messages':[{'role':'user','content':'What was the net income in 2023?'}]})
# print(r.json()[0]['messages'][-1]['content'])
#
# Path B — OpenAI ChatCompletion (mlflow.pyfunc.ChatModel / ChatAgent, v2 style):
# import openai
# client = openai.OpenAI(api_key=DATABRICKS_TOKEN, base_url=f'{DATABRICKS_HOST}/serving-endpoints')
# resp = client.chat.completions.create(model='<your-endpoint-name>',
#     messages=[{'role':'user','content':'What was the net income in 2023?'}])
# print(resp.choices[0].message.content)



## Part 3 — Client SDK demo
Instantiate `DocumentAnalystClient`, health-check, ask, stream, and show timeout/retry handling (Task 3.2).


In [0]:
# from client.sdk import DocumentAnalystClient
# c = DocumentAnalystClient(...)
# assert c.health_check() is True
# print(c.ask('What was the net income in 2023?'))



In [0]:
# ask_streaming demo
# for chunk in c.ask_streaming('Summarize FY2023 revenue.'): print(chunk, end='')



In [0]:
# Simulate timeout (timeout=0.001) and endpoint-unavailable retry behavior

